# `net_payout_yield` (shareholder yield) in the strategy lab

The engine is `rr_strategy_lab.ipynb` (repo root): section A below is copied from it unchanged (via `signals/net_operating_assets/noa_strategy_lab.ipynb`). Only the setup, signals (B), strategies (C) and run (D) differ.

**Question:** does a 40-stock, once-a-year, long-only portfolio of the highest net payout yields (dividends + buybacks − stock issued, ÷ market value; higher = better) beat the lab's equal-weighted universe and lift the Sharpe ratio, and does adding `net_payout_yield` to the lab's defensive composite help?

**Plan** (fixed 2026-11-24 before any return was computed; full text in `signals/shareholder_yield/backtest.md`):
- Lab settings unchanged: December formation, 12-month buy-and-hold, NYSE-median size, price >= $5, 40 names, 30% sector cap, FF3 + UMD alpha, design = returns through 2009-12, holdout after.
- New strategies: `SY_EW`, `SY_MOM_EW` (drop bottom-10% momentum), `SY_VW`, `SY_IND_EW` (yield ranked within FF12 sector), `DEF_SY_EW` (defensive composite + high yield), the control `DEF_SYELIG_EW` (defensive composite on the same eligible firms, no yield), and after the pick `<PICK>_GROSS` (gross `payout_yield`, the Step 1 named check) and `<PICK>_XPREF` (no preferred-stock adjustment, the Step 9 robustness column). `n_trials` = 10 + 8 = 18.
- Financial firms (SIC 6000-6999) and firms with common equity <= 0 are ineligible for the payout strategies (as in the paper and OSAP's replication); they stay in the universe. The signal sits on each firm's primary share class only.
- Pick = the strategy among `SY_EW`, `SY_MOM_EW`, `SY_VW`, `SY_IND_EW`, `DEF_SY_EW` with the best design-sample Sharpe. Look at the holdout once.

**Inputs (all under `data/`, never committed):** `crsp_monthly_ratios_clean.parquet`, `shareholder_yield/shareholder_yield_monthly.parquet` and `shareholder_yield/payout_events.parquet` (from `build_shareholder_yield.py`), `ff_factors_monthly.csv` (Ken French FF3 + momentum; downloaded if missing) and `ff_monthly_kf.csv` (FF5 + momentum, for one diagnostic).

**Before committing this notebook, clear all outputs** (they are computed from licensed CRSP and Compustat data).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize
from dataclasses import dataclass
from types import SimpleNamespace
from pathlib import Path
%matplotlib inline

USE_SYNTHETIC = False            # True = fake data, for smoke tests

# repo root = the first folder at or above the working directory that has data/crsp_monthly_ratios_clean.parquet
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'crsp_monthly_ratios_clean.parquet').exists()), None)
assert ROOT is not None or USE_SYNTHETIC, 'run from inside the repo (data/crsp_monthly_ratios_clean.parquet not found)'
DATA = (ROOT or Path.cwd()) / 'data'

CFG = dict(
    panel_path   = DATA / 'crsp_monthly_ratios_clean.parquet',
    factors_path = DATA / 'ff_factors_monthly.csv',   # cache; downloaded from Ken French if missing
    ff5_path     = DATA / 'ff_monthly_kf.csv',        # FF5 + momentum (percent), FF5 diagnostic only
    sy_path      = DATA / 'shareholder_yield' / 'shareholder_yield_monthly.parquet',
    sy_events    = DATA / 'shareholder_yield' / 'payout_events.parquet',
    factor_model = ['mktrf', 'smb', 'hml', 'umd'],
    etf_path     = DATA / 'etf_monthly_returns.csv',
    names_path   = None,

    form_month = 12,
    first_form = '1983-12-31',
    design_end = '2009-12-31',     # returns up to here = design sample; after = holdout
    horizon    = 12,

    # universe
    size_pct   = 0.50,
    min_price  = 5.0,
    universe_needs = ['vol36'],

    # portfolio defaults
    n_hold     = 40,
    sector_cap = 0.30,

    # min-variance weights
    cov_win = 60, shrink = 0.5, mv_cap = 0.05,

    n_trials = 18,                 # the lab's 10 + the 8 payout strategies below (control and the two reruns included)
)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

---
# A. Library
Run these once. They are pure functions: inputs come in as arguments and nothing reads globals except defaults.

### A1. Data loaders

In [ ]:
BASE_COLS = ['permno', 'date', 'primaryexch', 'siccd', 'mthret', 'prc', 'me', 'me_lag', 'gprof', 'debt_at', 'ep']
NAME_COLS = ['ticker', 'comnam', 'conm']
NUM_COLS  = ['mthret', 'prc', 'me', 'me_lag', 'gprof', 'debt_at', 'ep']

def load_panel(path, extra_cols=()):
    try:
        import pyarrow.parquet as pq
        avail = set(pq.read_schema(path).names)
    except ImportError:
        avail = set(pd.read_parquet(path).columns)
    missing = [c for c in BASE_COLS if c not in avail and c != 'siccd']
    assert not missing, f'missing columns: {missing}'
    want = [c for c in dict.fromkeys(BASE_COLS + list(extra_cols) + NAME_COLS) if c in avail]
    return clean_panel(pd.read_parquet(path, columns=want))

def clean_panel(df):
    df = df.copy()
    if 'siccd' not in df:
        df['siccd'] = np.nan
    df['permno'] = df['permno'].astype('int64')
    df['date'] = pd.to_datetime(df['date']) + pd.offsets.MonthEnd(0)
    for c in NUM_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')
    return df.sort_values(['permno', 'date']).reset_index(drop=True)

# ---- Fama-French factors from Ken French's data library (public; same series as ff.factors_monthly on WRDS)
FRENCH = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'

def _read_french_zip(fname):
    import io, zipfile, urllib.request
    raw = urllib.request.urlopen(FRENCH + fname, timeout=60).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        txt = z.read(z.namelist()[0]).decode('latin-1')
    header, rows = None, []
    for line in txt.splitlines():
        parts = [p.strip() for p in line.split(',')]
        is_month = len(parts[0]) == 6 and parts[0].isdigit()
        if is_month:
            rows.append(parts)
        elif rows:                       # monthly block ended (annual block follows)
            break
        elif len(parts) > 1 and parts[0] == '':
            header = parts[1:]
    df = pd.DataFrame([r[1:] for r in rows], columns=header,
                      index=pd.to_datetime([r[0] for r in rows], format='%Y%m') + pd.offsets.MonthEnd(0))
    df = df.apply(pd.to_numeric, errors='coerce')
    return df.mask(df <= -99.99) / 100            # -99.99 / -999 = missing

def load_ff_factors(cache='ff_factors_monthly.csv', refresh=False):
    if cache and Path(cache).exists() and not refresh:
        return pd.read_csv(cache, index_col=0, parse_dates=True)
    f3  = _read_french_zip('F-F_Research_Data_Factors_CSV.zip')
    mom = _read_french_zip('F-F_Momentum_Factor_CSV.zip')
    ff = f3.join(mom, how='left')
    ff.columns = [c.lower().replace('-', '') for c in ff.columns]
    ff = ff.rename(columns={'mom': 'umd'})[['mktrf', 'smb', 'hml', 'umd', 'rf']]
    ff.index.name = 'date'
    if cache:
        ff.to_csv(cache)
    return ff

def load_etf_returns(path=None, tickers=None, source='csv'):
    # 'csv': date + one column per ticker (monthly total returns, decimals)
    # 'yfinance': pulls adjusted closes and compounds to month-end returns (needs `pip install yfinance`)
    if source == 'yfinance':
        import yfinance as yf
        px = yf.download(tickers, start='2002-01-01', auto_adjust=True, progress=False)['Close']
        r = px.resample('ME').last().pct_change().dropna(how='all')
        if path:
            r.to_csv(path)
        return r
    if path is None or not Path(path).exists():
        return None
    r = pd.read_csv(path, index_col=0, parse_dates=True)
    r.index = r.index + pd.offsets.MonthEnd(0)
    return r.astype('float64')

def load_names(path):
    if path is None or not Path(path).exists():
        return None
    n = pd.read_csv(path)
    n['permno'] = n['permno'].astype('int64')
    return n.drop_duplicates('permno', keep='last')

In [ ]:
# ---- synthetic data, for testing the code without the real panel
def make_synthetic_panel(n_firms=1500, start='1975-01-31', end='2025-12-31', seed=0):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, end, freq='ME'); T = len(dates)
    mkt = rng.normal(0.007, 0.045, T)
    vol_i  = rng.lognormal(np.log(0.08), 0.45, n_firms)          # idiosyncratic monthly vol
    beta   = np.clip(0.6 + 4 * vol_i + rng.normal(0, 0.2, n_firms), 0.2, 2.5)
    gprof0 = rng.normal(0.3, 0.25, n_firms)
    debt0  = rng.beta(2, 6, n_firms)
    ep0    = rng.normal(0.04, 0.06, n_firms)
    # small "defensive" premium: profitable, low-debt, cheap names earn a bit more
    alpha = 0.002 * (gprof0 - 0.3) / 0.25 - 0.001 * (debt0 - 0.25) / 0.15 + 0.001 * (ep0 - 0.04) / 0.06
    R = beta[None, :] * mkt[:, None] + alpha[None, :] + vol_i[None, :] * rng.standard_normal((T, n_firms))
    R = np.maximum(R, -0.95)
    entry = rng.integers(-T // 2, T - 40, n_firms).clip(0)
    exit_ = np.minimum(entry + rng.integers(60, 2 * T, n_firms), T)
    t = np.arange(T)[:, None]
    alive = (t >= entry) & (t < exit_)
    me0 = rng.lognormal(np.log(800), 1.5, n_firms) * 1e3                  # $000s, as in CRSP
    me = me0 * np.cumprod(1 + np.where(alive, R, 0), axis=0)
    shares = me0 / rng.uniform(8, 80, n_firms)
    drift = lambda x0, s: x0 + np.cumsum(rng.normal(0, s, (T, n_firms)), axis=0) * 0.1
    sic_pool = np.array([2000, 2500, 3500, 1300, 2810, 3570, 4810, 4910, 5300, 2834, 6020, 7000, 1500])
    df = pd.DataFrame({
        'permno': np.tile(np.arange(10000, 10000 + n_firms), T),
        'date': np.repeat(dates, n_firms),
        'primaryexch': np.tile(np.where(rng.random(n_firms) < 0.4, 'N', 'Q'), T),
        'siccd': np.tile(rng.choice(sic_pool, n_firms), T).astype(float),
        'mthret': R.ravel(),
        'prc': (me / shares).ravel(),
        'me': me.ravel(),
        'me_lag': np.vstack([np.full(n_firms, np.nan), me[:-1]]).ravel(),
        'gprof': drift(gprof0, 0.02).ravel(),
        'debt_at': np.clip(drift(debt0, 0.01), 0, None).ravel(),
        'ep': drift(ep0, 0.005).ravel(),
        'ticker': np.tile([f'S{i:04d}' for i in range(n_firms)], T),
    })
    df = df[alive.ravel()]
    return clean_panel(df)

def make_synthetic_factors(crsp, seed=1):
    rng = np.random.default_rng(seed)
    vw = vw_market(crsp)
    ff = pd.DataFrame({'rf': 0.003}, index=vw.index)
    ff['mktrf'] = vw - ff['rf']
    for c in ['smb', 'hml', 'umd']:
        ff[c] = rng.normal(0.002, 0.03, len(ff))
    return ff

def make_synthetic_etfs(index, seed=2):
    rng = np.random.default_rng(seed)
    idx = index[index >= '2002-07-31']
    return pd.DataFrame({'IEF': rng.normal(0.003, 0.019, len(idx)),
                         'TLT': rng.normal(0.004, 0.038, len(idx)),
                         'GLD': rng.normal(0.007, 0.048, len(idx))}, index=idx)

### A2. Signal helpers and sectors
All per-`permno` operations work on **rows**, so a lag of 1 is the previous observation (the original notebook's convention). `add_momentum` checks for calendar gaps.

In [ ]:
def by_permno(crsp, col):
    return crsp.groupby('permno', sort=False)[col]

def roll(crsp, col, win, fn='mean', min_periods=None):
    # rolling stat per permno, aligned to crsp rows; fn = 'mean', 'std', 'sum', 'max', ...
    r = by_permno(crsp, col).rolling(win, min_periods=min_periods or win)
    return getattr(r, fn)().reset_index(level=0, drop=True)

def lag(crsp, col, k=1):
    return by_permno(crsp, col).shift(k)

def add_volatility(crsp, win=36, min_periods=24, name='vol36'):
    crsp[name] = roll(crsp, 'mthret', win, 'std', min_periods)
    return crsp

def add_momentum(crsp, skip=1, look=11, name='mom'):
    # cumulative return over months t-look-skip+1 ... t-skip (default 12-1), requiring no calendar gaps
    mi = crsp['date'].dt.year * 12 + crsp['date'].dt.month
    lr = np.log1p(lag(crsp, 'mthret', skip))
    s = lr.groupby(crsp['permno'], sort=False).rolling(look, min_periods=look).sum().reset_index(level=0, drop=True)
    gm = mi.groupby(crsp['permno'])
    span = gm.shift(skip) - gm.shift(skip + look - 1)
    gap = mi - gm.shift(skip)
    crsp[name] = np.where((span == look - 1) & (gap == skip), np.expm1(s), np.nan)
    return crsp

FF12 = {
    'NoDur': [(100, 999), (2000, 2399), (2700, 2749), (2770, 2799), (3100, 3199), (3940, 3989)],
    'Durbl': [(2500, 2519), (2590, 2599), (3630, 3659), (3710, 3711), (3714, 3714), (3716, 3716),
              (3750, 3751), (3792, 3792), (3900, 3939), (3990, 3999)],
    'Manuf': [(2520, 2589), (2600, 2699), (2750, 2769), (3000, 3099), (3200, 3569), (3580, 3629),
              (3700, 3709), (3712, 3713), (3715, 3715), (3717, 3749), (3752, 3791), (3793, 3799),
              (3830, 3839), (3860, 3899)],
    'Enrgy': [(1200, 1399), (2900, 2999)],
    'Chems': [(2800, 2829), (2840, 2899)],
    'BusEq': [(3570, 3579), (3660, 3692), (3694, 3699), (3810, 3829), (7370, 7379)],
    'Telcm': [(4800, 4899)],
    'Utils': [(4900, 4949)],
    'Shops': [(5000, 5999), (7200, 7299), (7600, 7699)],
    'Hlth':  [(2830, 2839), (3693, 3693), (3840, 3859), (8000, 8099)],
    'Money': [(6000, 6999)],
}

def add_ff12(crsp, name='sector'):
    sic = pd.to_numeric(crsp['siccd'], errors='coerce')
    out = np.full(len(crsp), 'Other', dtype=object)
    for sec, ranges in FF12.items():
        m = np.zeros(len(crsp), bool)
        for lo, hi in ranges:
            m |= sic.between(lo, hi).to_numpy()
        out[m] = sec
    crsp[name] = out
    return crsp

def vw_market(crsp):
    v = crsp.dropna(subset=['mthret', 'me_lag'])
    v = v[v['me_lag'] > 0]
    return (v['mthret'] * v['me_lag']).groupby(v['date']).sum() / v['me_lag'].groupby(v['date']).sum()

### A3. Universe, scoring helpers, selection, weights
- `form_universe` applies the screens (NYSE size percentile, price, `universe_needs`) at each formation date. The rows are one per stock-cohort and keep **every** `crsp` column, so score functions can use any signal.
- Score helpers: `rank(f, col, higher_better)` gives the percentile within the cohort. `composite(f, {...})` is the average rank (all inputs required). `mom_ok(f, pct)` is the falling-knife filter.
- `select` takes the top `n` by score, subject to a sector cap.
- A weight function has the signature `fn(sel, data, cfg) -> Series` aligned to `sel.index`. It does not need to sum to 1.

In [ ]:
def form_universe(crsp, form_month, cfg):
    f = crsp[crsp['date'].dt.month == form_month].dropna(subset=['me', 'prc', *cfg['universe_needs']])
    f = f[(f['prc'].abs() >= cfg['min_price']) & (f['date'] >= pd.Timestamp(cfg['first_form']))]
    bp = f[f['primaryexch'] == 'N'].groupby('date')['me'].quantile(cfg['size_pct']).rename('me_bp')
    f = f.merge(bp, on='date')
    f = f[f['me'] >= f['me_bp']]
    return f.rename(columns={'date': 'cohort'}).reset_index(drop=True)

# ---- scoring helpers (all return Series aligned to f)
def rank(f, col, higher_better=True):
    return f.groupby('cohort')[col].rank(pct=True, ascending=higher_better)

def composite(f, spec, weights=None):
    # spec = {'vol36': False, 'gprof': True, ...}; True means higher is better. NaN if any input missing.
    R = pd.concat({c: rank(f, c, hb) for c, hb in spec.items()}, axis=1)
    if weights is None:
        return R.mean(axis=1, skipna=False)
    w = pd.Series(weights, dtype=float).reindex(R.columns)
    return R.mul(w, axis=1).sum(axis=1, min_count=len(w)) / w.sum()

def mom_ok(f, pct=0.10, col='mom'):
    return rank(f, col) > pct           # NaN momentum -> False

# ---- selection
def select(f, score, n, cap=None):
    # top-n per cohort by score with at most floor(cap*n) names per sector (greedy, vectorized)
    d = f.assign(_score=score).dropna(subset=['_score'])
    d = d.sort_values(['cohort', '_score', 'permno'], ascending=[True, False, True])   # ties -> lower permno
    if cap is not None and cap < 1:
        max_per = max(1, int(np.floor(cap * n)))
        d = d[d.groupby(['cohort', 'sector']).cumcount() < max_per]
    return d.groupby('cohort').head(n)

# ---- weights
def w_equal(sel, data, cfg):
    return pd.Series(1.0, index=sel.index)

def w_invvol(sel, data, cfg, col='vol36'):
    return 1 / sel[col]

def w_value(sel, data, cfg):
    return sel['me']

def shrunk_cov(R, shrink):
    S = R.cov(min_periods=24)
    sd = np.sqrt(np.diag(S.fillna(0).to_numpy()))
    sd = np.where(sd > 0, sd, np.nanmedian(sd[sd > 0]))
    C = S.to_numpy() / np.outer(sd, sd)
    rbar = np.nanmean(C[~np.eye(len(C), dtype=bool)])
    C = np.where(np.isnan(C), rbar, C)
    np.fill_diagonal(C, 1.0)
    C = (1 - shrink) * C + shrink * (rbar + (1 - rbar) * np.eye(len(C)))
    Sig = C * np.outer(sd, sd)
    ev, V = np.linalg.eigh(Sig)
    return (V * np.maximum(ev, 1e-4 * ev.max())) @ V.T

def min_var_weights(cohort, permnos, data, cfg):
    start = cohort - pd.offsets.MonthEnd(cfg['cov_win'] - 1)
    R = data.ret_by_date.loc[start:cohort]
    R = R[R['permno'].isin(permnos)].pivot_table(index='date', columns='permno', values='mthret')
    Sig = shrunk_cov(R.reindex(columns=permnos), cfg['shrink'])
    n = len(permnos)
    cap = max(cfg['mv_cap'], 1.0 / n)
    res = optimize.minimize(lambda w: w @ Sig @ w, np.full(n, 1 / n), jac=lambda w: 2 * Sig @ w,
                            bounds=[(0, cap)] * n, constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}],
                            method='SLSQP', options={'maxiter': 500, 'ftol': 1e-12})
    w = np.clip(res.x, 0, None)
    return w / w.sum()

def w_minvar(sel, data, cfg):
    parts = [pd.Series(min_var_weights(c, d['permno'].tolist(), data, cfg), index=d.index)
             for c, d in sel.groupby('cohort')]
    return pd.concat(parts)

WEIGHTS = {'EW': w_equal, 'IV': w_invvol, 'VW': w_value, 'MV': w_minvar}

### A4. Strategy definition and buy-and-hold engine
Each position's value starts at `w0` and grows with its own returns over `horizon` months. A position drops out at its first missing return, and its value is redistributed pro rata.

In [ ]:
@dataclass
class Strategy:
    name: str
    score: object                 # fn(f) -> Series, higher = better, NaN = ineligible
    weight: object = 'EW'         # 'EW' | 'IV' | 'VW' | 'MV' | fn(sel, data, cfg) -> Series
    n: int = None                 # default cfg['n_hold']
    sector_cap: float = None      # default cfg['sector_cap']; 1.0 = no cap

def prepare(crsp, ff):
    return SimpleNamespace(
        crsp=crsp, ff=ff,
        ret_lookup=crsp[['permno', 'date', 'mthret']],
        ret_by_date=crsp.set_index('date')[['permno', 'mthret']].sort_index(),
        vw_mkt=vw_market(crsp),
    )

def build_holdings(f, strategies, data, cfg, add_ew_univ=True):
    parts = []
    for s in strategies:
        n = s.n or cfg['n_hold']
        cap = cfg['sector_cap'] if s.sector_cap is None else s.sector_cap
        sel = select(f, s.score(f), n, cap)
        wfn = WEIGHTS[s.weight] if isinstance(s.weight, str) else s.weight
        w = pd.Series(wfn(sel, data, cfg), index=sel.index).astype('float64')
        parts.append(sel[['cohort', 'permno']].assign(w0=w, port=s.name))
    if add_ew_univ:
        parts.append(f[['cohort', 'permno']].assign(w0=1.0, port='EW_UNIV'))
    h = pd.concat(parts, ignore_index=True)
    h['w0'] = h['w0'] / h.groupby(['port', 'cohort'])['w0'].transform('sum')
    return h

def bh_returns(holdings, ret_lookup, horizon=12):
    h = holdings.merge(pd.DataFrame({'k': np.arange(1, horizon + 1)}), how='cross')
    h['date'] = (h['cohort'].dt.to_period('M') + h['k']).dt.to_timestamp(how='end').dt.normalize()
    h = h.merge(ret_lookup, on=['permno', 'date'], how='left').sort_values(['port', 'cohort', 'permno', 'k'])
    key = [h['port'], h['cohort'], h['permno']]
    h['alive'] = h['mthret'].notna().astype(int).groupby(key).cumprod().astype(bool)
    r = h['mthret'].where(h['alive'], 0.0)
    growth = (1 + r).groupby(key).cumprod()
    h['v'] = (h['w0'] * growth / (1 + r)).where(h['alive'], 0.0)
    h['vr'] = h['v'] * r
    agg = h.groupby(['port', 'date']).agg(vr=('vr', 'sum'), v=('v', 'sum'))
    return (agg['vr'] / agg['v']).unstack('port')

def run(strategies, data, cfg, form_month=None):
    # returns (monthly returns: strategies + EW_UNIV + VW_MKT, holdings, formation universe)
    f = form_universe(data.crsp, form_month or cfg['form_month'], cfg)
    hold = build_holdings(f, strategies, data, cfg)
    rets = bh_returns(hold, data.ret_lookup, cfg['horizon']).dropna(subset=['EW_UNIV'])
    rets['VW_MKT'] = data.vw_mkt.reindex(rets.index)
    cols = [s.name for s in strategies] + ['EW_UNIV', 'VW_MKT']
    return rets[cols], hold, f

### A5. Evaluation
- **Sharpe SE:** the iid approximation, annualized.
- **`z vs <bench>`:** the Jobson-Korkie/Memmel test of the Sharpe-ratio difference.
- **alpha:** the factor model in `cfg['factor_model']`, with Newey-West t-statistics.

In [ ]:
def max_dd(r):
    w = (1 + r.fillna(0)).cumprod()
    return (w / w.cummax() - 1).min()

def nw_ols(y, X, lags=6):
    X = np.column_stack([np.ones(len(X)), X])
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    XtXi = np.linalg.inv(X.T @ X)
    S = (X * e[:, None]).T @ (X * e[:, None])
    for L in range(1, lags + 1):
        G = (X[L:] * e[L:, None]).T @ (X[:-L] * e[:-L, None])
        S += (1 - L / (lags + 1)) * (G + G.T)
    return b, b / np.sqrt(np.diag(XtXi @ S @ XtXi))

def jk_test(a, b):
    a, b = np.asarray(a, dtype='float64'), np.asarray(b, dtype='float64')
    T = len(a)
    s1, s2 = a.mean() / a.std(ddof=1), b.mean() / b.std(ddof=1)
    rho = np.corrcoef(a, b)[0, 1]
    var = (2 - 2 * rho + 0.5 * (s1**2 + s2**2 - 2 * s1 * s2 * rho**2)) / T
    return (s1 - s2) / np.sqrt(var)

def excess(R, ff):
    return R.sub(ff['rf'].reindex(R.index), axis=0)

def perf(R, ff, factors=('mktrf', 'smb', 'hml', 'umd'), bench='EW_UNIV'):
    R = R.loc[R.index.intersection(ff.dropna(subset=list(factors) + ['rf']).index)]
    ex = excess(R, ff)
    X = ff.loc[R.index, list(factors)].to_numpy()
    rows = {}
    for p in R.columns:
        e = ex[p].dropna()
        srm = e.mean() / e.std()
        b, t = nw_ols(ex[p].to_numpy(), X)
        rows[p] = {
            'ann ret %': 100 * ((1 + R[p]).prod() ** (12 / len(R)) - 1),
            'vol %': 100 * R[p].std() * np.sqrt(12),
            'Sharpe': srm * np.sqrt(12),
            'SE': np.sqrt((1 + 0.5 * srm**2) / len(e)) * np.sqrt(12),
            f'z vs {bench}': jk_test(ex[p], ex[bench]) if bench in R and p != bench else np.nan,
            'max DD %': 100 * max_dd(R[p]),
            'beta': b[1],
            'alpha %/yr': 1200 * b[0], 't(alpha)': t[0],
        }
    return pd.DataFrame(rows).T

def split(R, cfg):
    return {'FULL': R, 'DESIGN': R[R.index <= cfg['design_end']], 'HOLDOUT': R[R.index > cfg['design_end']]}

def report(R, data, cfg, periods=('DESIGN',)):
    # default: design sample only; pass periods=('HOLDOUT',) once you have made your choice
    for name in periods:
        P = split(R, cfg)[name]
        print(f'{name}: {P.index.min():%Y-%m} to {P.index.max():%Y-%m}, T = {len(P)}')
        display(perf(P, data.ff, cfg['factor_model']))

def plot_perf(R, ff, cfg, roll_win=60):
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
    W = (1 + R.fillna(0)).cumprod()
    np.log(W).plot(ax=ax[0], title='Cumulative log growth')
    ax[0].axvline(pd.Timestamp(cfg['design_end']), color='k', ls='--', lw=0.8)
    (W / W.cummax() - 1).plot(ax=ax[1], title='Drawdown', legend=False)
    plt.tight_layout(); plt.show()
    rs = excess(R, ff).rolling(roll_win).apply(lambda x: x.mean() / x.std() * np.sqrt(12), raw=True)
    rs.plot(figsize=(15, 3.5), title=f'Rolling {roll_win}-month Sharpe ratio')
    plt.axhline(0, color='k', lw=0.8); plt.show()

def deflated_sharpe(ex, sr_trials_m, n_trials):
    # Bailey & Lopez de Prado (2014). sr_trials_m: monthly Sharpe ratios of the trials
    ex = ex.dropna()
    sr, T = ex.mean() / ex.std(), len(ex)
    sk, ku = stats.skew(ex), stats.kurtosis(ex, fisher=False)
    em = 0.5772156649
    sr0 = np.sqrt(np.var(sr_trials_m, ddof=1)) * ((1 - em) * stats.norm.ppf(1 - 1 / n_trials)
                                                  + em * stats.norm.ppf(1 - 1 / (n_trials * np.e)))
    z = (sr - sr0) * np.sqrt(T - 1) / np.sqrt(1 - sk * sr + (ku - 1) / 4 * sr**2)
    return stats.norm.cdf(z), sr0 * np.sqrt(12)

def dsr_table(R, ff, strat_names, n_trials):
    ex = excess(R[strat_names], ff)
    sr_m = (ex.mean() / ex.std()).to_numpy()
    out = pd.DataFrame({p: dict(zip(['DSR', 'SR hurdle (ann.)'], deflated_sharpe(ex[p], sr_m, n_trials)))
                        for p in strat_names}).T
    out['Sharpe'] = sr_m * np.sqrt(12)
    return out

def name_turnover(hold, strat_names):
    out = {}
    for p, d in hold[hold['port'].isin(strat_names)].groupby('port'):
        sets = d.groupby('cohort')['permno'].apply(set).sort_index()
        out[p] = np.mean([1 - len(a & b) / len(b) for a, b in zip(sets.iloc[:-1], sets.iloc[1:])])
    return pd.Series(out, name='share of names replaced per rebalance')

def weight_summary(hold, strat_names):
    h = hold[hold['port'].isin(strat_names)]
    g = h.groupby(['port', 'cohort'])['w0']
    return pd.DataFrame({'names (median)': g.size().groupby('port').median(),
                         'max weight (median)': g.max().groupby('port').median()})

### A6. Robustness, multi-asset, and current portfolio

In [ ]:
def by_formation_month(strategies, data, cfg, months=range(1, 13)):
    # Sharpe per formation month + the staggered portfolio (average across cohorts formed in each month)
    names = [s.name for s in strategies] + ['EW_UNIV']
    sharpe, stag = {}, []
    for fm in months:
        r, _, _ = run(strategies, data, cfg, form_month=fm)
        ex = excess(r[names], data.ff)
        sharpe[fm] = ex.mean() / ex.std() * np.sqrt(12)
        stag.append(r[names])
    stag = pd.concat(stag).groupby(level=0).mean()
    stag = stag[stag.index >= pd.Timestamp(cfg['first_form']) + pd.offsets.MonthEnd(12)]
    ex = excess(stag, data.ff)
    stag_tbl = pd.DataFrame({'Sharpe': ex.mean() / ex.std() * np.sqrt(12),
                             'z vs EW': [jk_test(ex[p], ex['EW_UNIV']) if p != 'EW_UNIV' else np.nan
                                         for p in ex.columns]})
    return pd.DataFrame(sharpe).T.rename_axis('formation month'), stag_tbl

def bh_mix(R, W, horizon=12):
    # R: monthly sleeve returns; W: initial weights per cohort (index = formation dates)
    out = []
    for c, w in W.iterrows():
        dates = pd.date_range(c + pd.offsets.MonthEnd(1), periods=horizon, freq='ME')
        r = R.reindex(dates)[w.index].fillna(0.0)
        v = w.to_numpy() * np.vstack([np.ones(len(w)), (1 + r).cumprod().to_numpy()[:-1]])
        out.append(pd.Series((v * r.to_numpy()).sum(1) / v.sum(1), index=dates))
    return pd.concat(out)

def multi_asset(rets, etf_r, ff, stock_sleeve, etfs=('IEF', 'GLD'), start='2006-12-31', vol_win=36):
    S = pd.concat([rets[[stock_sleeve, 'EW_UNIV']], etf_r[list(etfs)]], axis=1)
    cohorts = list(pd.date_range(start, rets.index.max() - pd.offsets.MonthEnd(1), freq='12ME'))
    vol = S.rolling(vol_win, min_periods=24).std()
    W_rp = 1 / vol.loc[cohorts, [stock_sleeve, *etfs]]
    W_rp = W_rp.div(W_rp.sum(axis=1), axis=0)
    W_6040 = pd.DataFrame({'EW_UNIV': 0.6, etfs[0]: 0.4}, index=cohorts)
    M = pd.DataFrame({'MULTI_RP': bh_mix(S, W_rp), '60/40': bh_mix(S, W_6040)})
    M = M.join(S[[stock_sleeve, 'EW_UNIV', *etfs]]).dropna()
    ex = excess(M, ff)
    tbl = pd.DataFrame({
        'ann ret %': 100 * ((1 + M).prod() ** (12 / len(M)) - 1),
        'vol %': 100 * M.std() * np.sqrt(12),
        'Sharpe': ex.mean() / ex.std() * np.sqrt(12),
        'z vs EW': [jk_test(ex[c], ex['EW_UNIV']) if c != 'EW_UNIV' else np.nan for c in M.columns],
        'max DD %': [100 * max_dd(M[c]) for c in M.columns],
    })
    return M, tbl, W_rp

def current_portfolio(strategy, data, cfg, names=None, show_cols=('vol36', 'gprof', 'debt_at', 'ep', 'me', 'prc')):
    # forms the strategy on the latest month in the panel (same screens/score/cap/weights as the backtest)
    latest = data.crsp['date'].max()
    f_now = form_universe(data.crsp, latest.month, cfg)
    f_now = f_now[f_now['cohort'] == latest]
    print(f'Formation date: {latest:%Y-%m-%d} | eligible: {len(f_now)} | scored: {strategy.score(f_now).notna().sum()}')
    if latest.month != cfg['form_month']:
        print(f'Note: strategy forms in month {cfg["form_month"]}; this is a preview on the latest month.')
    f_now = f_now.assign(score=strategy.score(f_now))
    h = build_holdings(f_now, [strategy], data, cfg, add_ew_univ=False)
    rec = h.merge(f_now, on=['cohort', 'permno'])
    if names is not None:
        rec = rec.drop(columns=[c for c in names.columns if c in rec and c != 'permno']).merge(names, on='permno', how='left')
    id_cols = [c for c in ['ticker', 'company', 'comnam', 'conm'] if c in rec]
    rec = rec.sort_values('score', ascending=False).reset_index(drop=True)
    rec.index += 1
    out = rec[['permno', *id_cols, 'sector', 'w0', 'score', *[c for c in show_cols if c in rec]]].copy()
    out['w0'] *= 100
    out = out.rename(columns={'w0': 'weight %'})

    # changes vs last year's portfolio formed in the same month
    prev = latest - pd.offsets.MonthEnd(12)
    f_prev = form_universe(data.crsp, prev.month, cfg)
    f_prev = f_prev[f_prev['cohort'] == prev]
    if len(f_prev):
        p = set(select(f_prev, strategy.score(f_prev), strategy.n or cfg['n_hold'],
                       cfg['sector_cap'] if strategy.sector_cap is None else strategy.sector_cap)['permno'])
        c = set(out['permno'])
        print(f'vs {prev:%Y-%m}: keep {len(c & p)}, buy {len(c - p)}, sell {len(p - c)}')
        if id_cols:
            print('new buys:', sorted(out.loc[out['permno'].isin(c - p), id_cols[0]].dropna().astype(str)))
    return out, f_now

---
# B. Data and signals
The lab's signals (`vol36`, `mom`, sectors), plus `net_payout_yield`, the named check `payout_yield`, the robustness column `net_payout_yield_xpref`, the flags `financial`, `ceq_nonpos` and `zero_filled`, and the CRSP firm id `permco`, merged by `permno` and month. At a December formation the yields use the latest 10-K public by that month-end (at most 18 months after its fiscal year-end) and the firm's market value that month. The `_el` columns are the same values with ineligible firms blanked (financial firms and common equity <= 0): a stock without them (ineligible, no yield, secondary share class, ...) stays in the universe but cannot be picked by the payout strategies.

In [ ]:
SY_COLS = ['net_payout_yield', 'payout_yield', 'net_payout_yield_xpref']

if USE_SYNTHETIC:
    crsp = make_synthetic_panel()
    rng = np.random.default_rng(3)
    n = len(crsp)
    gross = np.where(rng.random(n) < 0.6, rng.lognormal(np.log(0.03), 0.8, n), 0.0)
    iss = np.where(rng.random(n) < 0.5, rng.lognormal(np.log(0.01), 1.0, n), 0.0)
    has = rng.random(n) < 0.92
    crsp['payout_yield'] = np.where(has, gross, np.nan)
    crsp['net_payout_yield'] = np.where(has, gross - iss, np.nan)
    crsp['net_payout_yield_xpref'] = crsp['net_payout_yield'] - np.where(rng.random(n) < 0.1, 0.005, 0.0)
    crsp['financial'] = pd.to_numeric(crsp['siccd'], errors='coerce').between(6000, 6999)
    crsp['ceq_nonpos'] = rng.random(n) < 0.04
    crsp['zero_filled'] = np.where(rng.random(n) < 0.1, 'sstk', '')
    crsp['permco'] = crsp['permno']
    crsp['gvkey'] = crsp['permno'].astype(str).astype('string')
    crsp['period_end'] = crsp['date'] - pd.offsets.MonthEnd(6)
    crsp['published_date'] = crsp['date'] - pd.offsets.MonthEnd(2)
    crsp['date_source'] = 'exact'
    ev = pd.DataFrame({'gvkey': crsp['gvkey'], 'period_end': crsp['period_end'],
                       'dvc': rng.random(n), 'prstkc': rng.random(n), 'sstk': rng.random(n)}).drop_duplicates(['gvkey', 'period_end'])
    ev['period_end'] = ev['period_end'].astype('datetime64[ns]')
else:
    crsp = load_panel(CFG['panel_path'])
    sc = ['permno', 'gvkey', 'date', *SY_COLS, 'financial', 'ceq_nonpos', 'zero_filled', 'permco',
          'period_end', 'published_date', 'date_source']
    sy = pd.read_parquet(CFG['sy_path'], columns=sc)
    sy['permno'] = sy['permno'].astype('int64')
    sy['date'] = pd.to_datetime(sy['date']) + pd.offsets.MonthEnd(0)
    for c in SY_COLS:
        sy[c] = pd.to_numeric(sy[c], errors='coerce').astype('float64')
    for c in ['period_end', 'published_date']:
        sy[c] = pd.to_datetime(sy[c]).astype('datetime64[ns]')
    sy['gvkey'] = sy['gvkey'].astype('string')
    crsp = crsp.merge(sy, on=['permno', 'date'], how='left', validate='one_to_one')
    ev = pd.read_parquet(CFG['sy_events'], columns=['gvkey', 'period_end', 'dvc', 'prstkc', 'sstk'])
    ev['period_end'] = pd.to_datetime(ev['period_end']).astype('datetime64[ns]')
    ev['gvkey'] = ev['gvkey'].astype('string')
crsp['financial'] = crsp['financial'].fillna(False).astype(bool)
crsp['ceq_nonpos'] = crsp['ceq_nonpos'].astype('boolean').fillna(False).astype(bool)
crsp = crsp.sort_values(['permno', 'date']).reset_index(drop=True)

eligible = ~crsp['financial'] & ~crsp['ceq_nonpos']
for c in SY_COLS:
    crsp[c + '_el'] = crsp[c].where(eligible)            # financial firms and ceq <= 0 ineligible

crsp = add_volatility(crsp, win=36, min_periods=24, name='vol36')
crsp = add_momentum(crsp, name='mom')
crsp = add_ff12(crsp)

if USE_SYNTHETIC:
    ff = make_synthetic_factors(crsp)
    rng5 = np.random.default_rng(4)
    ff5 = ff.assign(rmw=rng5.normal(0.003, 0.02, len(ff)), cma=rng5.normal(0.002, 0.02, len(ff)))
else:
    ff = load_ff_factors(CFG['factors_path'])
    ff5 = pd.read_csv(CFG['ff5_path'])
    ff5.index = pd.to_datetime(ff5['yyyymm'].astype(str), format='%Y%m') + pd.offsets.MonthEnd(0)
    ff5 = ff5.drop(columns='yyyymm') / 100
    ff5.index.name = 'date'

data = prepare(crsp, ff)
print(crsp.shape, '|', crsp['date'].min().date(), '->', crsp['date'].max().date(),
      '| factors through', ff.index.max().date(), '| FF5 through', ff5.index.max().date())
dec = crsp[crsp['date'].dt.month == 12]
print('share of December rows with net_payout_yield:', round(dec['net_payout_yield'].notna().mean(), 3),
      '| financial:', round(dec['financial'].mean(), 3), '| ceq <= 0:', round(dec['ceq_nonpos'].mean(), 3),
      '| eligible (net_payout_yield_el):', round(dec['net_payout_yield_el'].notna().mean(), 3))
crsp[['vol36', 'mom', 'gprof', 'debt_at', 'ep', *SY_COLS, 'net_payout_yield_el']].describe().T

---
# C. Strategies
The lab's four baselines (already counted in its 10 trials), the five payout strategies and the control fixed in the plan. `sy_strategies(col)` builds them for `net_payout_yield_el` (the signal), or, after the pick, for `payout_yield_el` (`_GROSS`) or `net_payout_yield_xpref_el` (`_XPREF`).

In [ ]:
DEF_SPEC = {'vol36': False, 'gprof': True, 'debt_at': False, 'ep': True}   # True = higher is better

def s_lowvol(f):
    return rank(f, 'vol36', higher_better=False).where(mom_ok(f, 0.10))

def s_def(f):
    return composite(f, DEF_SPEC).where(mom_ok(f, 0.10))

BASELINES = [
    Strategy('LOWVOL_EW', s_lowvol, 'EW'),
    Strategy('DEF_EW',    s_def,    'EW'),
    Strategy('DEF_IV',    s_def,    'IV'),
    Strategy('DEF_MV',    s_def,    'MV'),
]

# ---- payout-yield ideas (higher yield = better; the _el columns are NaN for financial firms, ceq <= 0,
# firms without a yield and secondary share classes, so they are ineligible)
def sy_strategies(col='net_payout_yield_el', suffix=''):
    def s_sy(f):                               # higher net payout yield = better
        return rank(f, col, higher_better=True)

    def s_sy_mom(f):                           # same, without the bottom 10% on momentum
        return s_sy(f).where(mom_ok(f, 0.10))

    def s_sy_ind(f):                           # yield ranked within the FF12 sector of each cohort
        return f.groupby(['cohort', 'sector'])[col].rank(pct=True, ascending=True)

    spec = {**DEF_SPEC, col: True}

    def s_def_sy(f):                           # defensive composite + high yield, equal ranks
        return composite(f, spec).where(mom_ok(f, 0.10))

    return {
        'SY_EW':     Strategy('SY_EW' + suffix,     s_sy,     'EW'),
        'SY_MOM_EW': Strategy('SY_MOM_EW' + suffix, s_sy_mom, 'EW'),
        'SY_VW':     Strategy('SY_VW' + suffix,     s_sy,     'VW'),
        'SY_IND_EW': Strategy('SY_IND_EW' + suffix, s_sy_ind, 'EW'),
        'DEF_SY_EW': Strategy('DEF_SY_EW' + suffix, s_def_sy, 'EW'),
    }

def s_def_syelig(f):                           # control: defensive composite on the same eligible firms, no yield
    return composite(f, DEF_SPEC).where(mom_ok(f, 0.10) & f['net_payout_yield_el'].notna())

SY_IDEAS = list(sy_strategies('net_payout_yield_el').values())
CONTROLS = [Strategy('DEF_SYELIG_EW', s_def_syelig, 'EW')]

STRATEGIES = BASELINES + SY_IDEAS + CONTROLS
SY_NAMES = [s.name for s in SY_IDEAS]          # pick candidates (the control is not one)

---
# D. Run and evaluate
### Design sample

In [ ]:
rets, hold, fu = run(STRATEGIES, data, CFG)
NAMES = [s.name for s in STRATEGIES]
print('universe per cohort:', fu.groupby('cohort').size().describe()[['min', '50%', 'max']].to_dict())
print('eligible (net_payout_yield_el) per cohort:', fu.groupby('cohort')['net_payout_yield_el'].count().describe()[['min', '50%', 'max']].to_dict())
print(rets.index.min().date(), '->', rets.index.max().date(), '| months:', len(rets))
display(weight_summary(hold, NAMES))

### Checks before reading the numbers
- `SY_EW` selection recomputed with a plain loop for every cohort (sort by yield, skip a sector once it has 12 names, stop at 40): must be identical.
- Buy-and-hold returns of `SY_EW` recomputed with a plain loop for six portfolio-years: must agree to rounding.
- Every payout portfolio: 40 names per cohort, at most 12 per sector, no firm (`permco`) twice, every name has the signal, none financial or with common equity <= 0.
- Timing: every pick's 10-K public by its formation date, fiscal year ended within 18 months; share of exact dates.

In [ ]:
h_sy = hold[hold['port'].isin(SY_NAMES + ['DEF_SYELIG_EW'])].merge(
    fu[['cohort', 'permno', 'permco', 'sector', 'financial', 'ceq_nonpos', 'net_payout_yield_el',
        'period_end', 'published_date', 'date_source']], on=['cohort', 'permno'], how='left')

# 1. SY_EW selection by a plain loop
cap_n = max(1, int(np.floor(CFG['sector_cap'] * CFG['n_hold'])))
loop_ok = True
for c, g in fu.groupby('cohort'):
    g = g.dropna(subset=['net_payout_yield_el']).sort_values(['net_payout_yield_el', 'permno'], ascending=[False, True])
    picks, per = [], {}
    for _, r in g.iterrows():
        if per.get(r['sector'], 0) < cap_n:
            picks.append(r['permno']); per[r['sector']] = per.get(r['sector'], 0) + 1
        if len(picks) == CFG['n_hold']:
            break
    loop_ok &= set(picks) == set(hold.loc[(hold['port'] == 'SY_EW') & (hold['cohort'] == c), 'permno'])
print('SY_EW selection = plain loop, every cohort:', loop_ok)

# 2. buy-and-hold returns by a plain loop, six portfolio-years
rl = data.ret_lookup.set_index(['permno', 'date'])['mthret']
cohorts = sorted(hold['cohort'].unique())
worst = 0.0
for c in [cohorts[i] for i in np.linspace(0, len(cohorts) - 2, 6).astype(int)]:
    h = hold[(hold['port'] == 'SY_EW') & (hold['cohort'] == c)]
    v = dict(zip(h['permno'], h['w0']))
    for k in range(1, CFG['horizon'] + 1):
        d = (pd.Timestamp(c).to_period('M') + k).to_timestamp(how='end').normalize()
        num = den = 0.0
        for p in list(v):
            r = rl.get((p, d), np.nan)
            if np.isnan(r):
                v.pop(p); continue
            num += v[p] * r; den += v[p]; v[p] *= 1 + r
        if d in rets.index:
            worst = max(worst, abs(num / den - rets.loc[d, 'SY_EW']))
print('buy-and-hold loop vs engine, max |difference|:', worst)

# 3. portfolio composition rules
g = h_sy.groupby(['port', 'cohort'])
print('names per cohort (min, max):', g.size().min(), g.size().max(),
      '| max per sector:', h_sy.groupby(['port', 'cohort', 'sector']).size().max(),
      '| max positions per firm (permco):', h_sy.groupby(['port', 'cohort', 'permco']).size().max())
picks = h_sy[h_sy['port'].isin(SY_NAMES)]
print('picks without the signal:', picks['net_payout_yield_el'].isna().sum(),
      '| financial:', picks['financial'].sum(), '| ceq <= 0:', picks['ceq_nonpos'].sum())

# 4. timing
late = (picks['published_date'] > picks['cohort']).sum()
old = (picks['cohort'] > picks['period_end'] + pd.offsets.MonthEnd(18)).sum()
print(f'picks: {len(picks):,}; 10-K not yet public at formation: {late}; fiscal year older than 18 months: {old}; '
      f'exact dates: {(picks["date_source"] == "exact").mean():.1%}')

In [ ]:
report(rets, data, CFG)

In [ ]:
plot_perf(rets[rets.index <= CFG['design_end']], data.ff, CFG)

In [ ]:
D = rets[rets.index <= CFG['design_end']]
for k in (CFG['n_trials'], 30):
    print(f'deflated Sharpe ratio, n_trials = {k}')
    display(dsr_table(D, data.ff, NAMES, k))
name_turnover(hold, NAMES)

### Diagnostics (design sample; not trials)
- **Sector mix of `SY_EW`:** average share of the 40 names per FF12 sector, and the share of cohorts in which the sector is at the cap (12 names).
- **What `SY_EW` holds:** dividend payers, buyers-back, stock issuers and net issuers among the picks; zero-filled items; picks whose preferred-stock change is over half the yield; median yield of the picks vs the eligible universe.
- **Factor loadings:** full FF3 + UMD regression (Newey-West 6) for the payout strategies, then **FF5 + UMD** (adds profitability RMW and investment CMA).
- **Overlap with `DEF_EW`:** names shared and the correlation of excess returns.

In [ ]:
mix = h_sy[h_sy['port'] == 'SY_EW'].groupby(['cohort', 'sector']).size().unstack(fill_value=0)
display(pd.DataFrame({'avg share of names': mix.div(mix.sum(axis=1), axis=0).mean(),
                      'share of cohorts at cap': (mix >= cap_n).mean()}).sort_values('avg share of names', ascending=False))

pe = picks[picks['port'] == 'SY_EW'].merge(fu[['cohort', 'permno', 'gvkey', 'net_payout_yield', 'payout_yield',
                                              'net_payout_yield_xpref', 'zero_filled']], on=['cohort', 'permno'], how='left')
pe = pe.merge(ev, on=['gvkey', 'period_end'], how='left')
pref_part = pe['net_payout_yield'] - pe['net_payout_yield_xpref']
el = fu.dropna(subset=['net_payout_yield_el'])
display(pd.Series({
    'dividend payers (dvc > 0)': (pe['dvc'] > 0).mean(),
    'buyers-back (prstkc > 0)': (pe['prstkc'] > 0).mean(),
    'stock issuers (sstk > 0)': (pe['sstk'] > 0).mean(),
    'net issuers (net yield < 0)': (pe['net_payout_yield'] < 0).mean(),
    'any item zero-filled': (pe['zero_filled'].fillna('') != '').mean(),
    'preferred change > half the yield': (pref_part > 0.5 * pe['net_payout_yield']).mean(),
    'median net yield, picks': pe['net_payout_yield'].median(),
    'median net yield, eligible universe': el['net_payout_yield_el'].median(),
    'min net yield, picks': pe['net_payout_yield'].min(),
}, name='SY_EW picks, all design + holdout cohorts').to_frame())

def loadings(R, ff, factors):
    R = R.loc[R.index.intersection(ff.dropna(subset=list(factors) + ['rf']).index)]
    ex = excess(R, ff)
    X = ff.loc[R.index, list(factors)].to_numpy()
    out = {}
    for p in R.columns:
        b, t = nw_ols(ex[p].to_numpy(), X)
        out[p] = {'alpha %/yr': 1200 * b[0], 't(alpha)': t[0],
                  **{c: b[i + 1] for i, c in enumerate(factors)},
                  **{f't({c})': t[i + 1] for i, c in enumerate(factors)}}
    return pd.DataFrame(out).T

DIAG = SY_NAMES + ['DEF_SYELIG_EW', 'DEF_EW', 'EW_UNIV']
print('FF3 + UMD'); display(loadings(D[DIAG], data.ff, CFG['factor_model']))
FF5 = ['mktrf', 'smb', 'hml', 'rmw', 'cma', 'umd']
print('FF5 + UMD (diagnostic)'); display(loadings(D[DIAG], ff5, FF5))

a = hold[hold['port'] == 'SY_EW'][['cohort', 'permno']]
b = hold[hold['port'] == 'DEF_EW'][['cohort', 'permno']]
print(f"SY_EW names also in DEF_EW: {len(a.merge(b)) / len(a):.1%}; "
      f"correlation of excess returns (design): {excess(D[['SY_EW', 'DEF_EW']], data.ff).corr().iloc[0, 1]:.2f}")

### Pick (design sample only), named checks, and the `_GROSS` / `_XPREF` reruns
The pick is the payout strategy with the highest design-sample Sharpe ratio. Named checks (z = Jobson-Korkie test of the Sharpe difference): `DEF_SY_EW` vs `DEF_SYELIG_EW` (does the yield help, holding the firms fixed?) and `DEF_SY_EW` vs `DEF_EW` (the lab's strategy). Then the pick is rerun with `payout_yield` (`<PICK>_GROSS`, the named check) and with `net_payout_yield_xpref` (`<PICK>_XPREF`, robustness): not pick candidates.

In [ ]:
exD = excess(D[SY_NAMES], data.ff)
sr_design = exD.mean() / exD.std() * np.sqrt(12)
PICK = sr_design.idxmax()
print(sr_design.round(3).to_string())
print('pick:', PICK)
display(perf(D[['DEF_SY_EW', 'DEF_SYELIG_EW']], data.ff, CFG['factor_model'], bench='DEF_SYELIG_EW'))
display(perf(D[['DEF_SY_EW', 'DEF_EW']], data.ff, CFG['factor_model'], bench='DEF_EW'))

RERUNS = [sy_strategies('payout_yield_el', suffix='_GROSS')[PICK],
          sy_strategies('net_payout_yield_xpref_el', suffix='_XPREF')[PICK]]
rets_rr, hold_rr, _ = run(RERUNS, data, CFG)
for s in RERUNS:
    rets[s.name] = rets_rr[s.name].reindex(rets.index)
D = rets[rets.index <= CFG['design_end']]
for s in RERUNS:
    print(f'{PICK} vs {s.name} (design):')
    display(perf(D[[s.name, PICK]], data.ff, CFG['factor_model'], bench=PICK))
    ov = hold[hold['port'] == PICK].merge(hold_rr.loc[hold_rr['port'] == s.name, ['cohort', 'permno']], on=['cohort', 'permno'])
    print(f'names shared by {PICK} and {s.name}: {len(ov) / hold[hold["port"] == PICK].shape[0]:.1%}')

### Holdout: look once

In [ ]:
report(rets, data, CFG, periods=('HOLDOUT', 'FULL'))
H = rets[rets.index > CFG['design_end']]
print('named checks, holdout: DEF_SY_EW vs DEF_SYELIG_EW, and vs DEF_EW')
display(perf(H[['DEF_SY_EW', 'DEF_SYELIG_EW']], data.ff, CFG['factor_model'], bench='DEF_SYELIG_EW'))
display(perf(H[['DEF_SY_EW', 'DEF_EW']], data.ff, CFG['factor_model'], bench='DEF_EW'))
for s in RERUNS:
    print(f'{PICK} vs {s.name} (holdout):')
    display(perf(H[[s.name, PICK]], data.ff, CFG['factor_model'], bench=PICK))
print('FF5 + UMD (diagnostic), holdout'); display(loadings(H[DIAG], ff5, FF5))
print(f"correlation of SY_EW and DEF_EW excess returns (holdout): {excess(H[['SY_EW', 'DEF_EW']], data.ff).corr().iloc[0, 1]:.2f}")
plot_perf(rets.drop(columns=[s.name for s in RERUNS]), data.ff, CFG)

### Robustness: formation month and staggered cohorts
12 runs of the whole pipeline (MV excluded, as in the lab).

In [ ]:
RUN_ROBUSTNESS = True
if RUN_ROBUSTNESS:
    robust_strats = [s for s in STRATEGIES if s.weight != 'MV']
    sharpe_by_fm, stag_tbl = by_formation_month(robust_strats, data, CFG)
    print('Sharpe by formation month'); display(sharpe_by_fm)
    print('Staggered portfolio'); display(stag_tbl)

### Current portfolio (optional)
Off by default: the output lists stocks with CRSP prices and market values, so clear it before committing. Tickers are not in the panel, so this shows PERMNOs unless `CFG['names_path']` points to a `permno,ticker,company` CSV. Research only, not a recommendation.

In [ ]:
SHOW_CURRENT = False
if SHOW_CURRENT:
    pick = next(s for s in STRATEGIES if s.name == PICK)
    now, f_now = current_portfolio(pick, data, CFG, names=load_names(CFG['names_path']),
                                   show_cols=('net_payout_yield', 'payout_yield', 'mom', 'vol36', 'me', 'prc'))
    pd.set_option('display.max_rows', 60)
    display(now.round(3))